# 01 · F3: 피드백 + 복습 + easy 보충 재학습
seed 42/43/44마다 6번 재학습하고, 각 모델로 아직 학습하지 않은 다음 window를 예측합니다. 운영과 같은 버전 규칙(update 4~6은 이 arm의 2.0에서 시작)과 3 epoch를 씁니다. 연결이 끊기면 같은 셀을 다시 실행하면 이어서 합니다.

## 공통 준비
개인 `alexyoo0506@gmail.com` Colab Pro와 해당 계정의 Drive를 사용합니다. Drive 연결 시 해당 계정을 선택하고 Google 화면에서 직접 승인합니다. GCP·Vertex 제출, GCS 업로드, 운영 `CURRENT` 변경 코드는 없습니다.

제공한 실행 패키지의 `retraining-effect-f3` 폴더를 `MyDrive/LLM_Router/` 아래에 업로드하세요. 첫 실험 폴더 `retraining-effect`의 로거 결과와 run001 결과를 읽기 전용으로 재사용하므로 그 폴더는 그대로 둡니다. 입력 데이터와 R0(운영 1.0) 모델은 **R1→R0 전이 실험 때 올린 `MyDrive/LLM_Router/transfer-experiments/inputs/experiment-A`를 읽기 전용으로 재사용**합니다. 경로·SHA가 모두 채워져 있으므로 수정 없이 전체 실행합니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile

BASE = Path("/content/drive/MyDrive/LLM_Router/retraining-effect-f3")
FIRST = Path("/content/drive/MyDrive/LLM_Router/retraining-effect")  # 첫 실험: 로거·F1/F2 결과 재사용(읽기 전용)
SOURCE_ZIP = BASE / "source" / "retraining-source-f3.zip"
EXPECTED_SOURCE_SHA256 = SOURCE_ZIP.with_suffix(".sha256").read_text().split()[0]
RUN_ID = "retraining-effect-v2-f3-run001"
RUN = BASE / "runs" / RUN_ID
LOG = FIRST / "logger" / "logger-log.parquet"
REFERENCE = FIRST / "runs" / "retraining-effect-v1-run001"
# R1->R0 전이 실험 때 올린 입력과 복구된 anchor 모델을 읽기 전용으로 재사용한다.
INPUTS = Path("/content/drive/MyDrive/LLM_Router/transfer-experiments/inputs/experiment-A")
PINS = INPUTS / "source_pins.json"
DATA = INPUTS / "data"

assert len(EXPECTED_SOURCE_SHA256) == 64, "소스 ZIP의 SHA-256 파일이 필요합니다"
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == EXPECTED_SOURCE_SHA256
REPO = Path("/content/router-retraining-source")
with zipfile.ZipFile(SOURCE_ZIP) as z:
    for name in z.namelist():
        assert (REPO / name).resolve().is_relative_to(REPO.resolve()), name
    z.extractall(REPO)
for name, expected in json.loads((REPO / "source-bundle-manifest.json").read_text()).items():
    assert hashlib.sha256((REPO / name).read_bytes()).hexdigest() == expected, name
EXPERIMENT = REPO / "ml/notebooks/retraining_effect"
HELPER = EXPERIMENT / "experiment.py"
CONFIG = EXPERIMENT / "experiment_config_f3.json"
ENV = dict(os.environ, PYTHONPATH=str(REPO / "ml/src"))
def run(*args):
    subprocess.run([sys.executable, str(HELPER), *map(str, args)], env=ENV, check=True)
assert PINS.is_file(), f"전이 실험 입력이 필요합니다: {PINS}"
assert LOG.is_file() and (REFERENCE / "plan.json").is_file(), "첫 실험 retraining-effect 폴더가 필요합니다"
print("Run directory:", RUN)

## 패키지 설치
Colab 메뉴 **런타임 → 런타임 유형 변경 → 런타임 버전 2026.07**(Python 3.12.13), GPU **T4**를 선택합니다. 설치 후 이미 import한 패키지 버전이 다르면 세션을 재시작하고 경로 설정 셀부터 다시 실행하세요.

In [ ]:
assert sys.version_info[:2] in {(3, 11), (3, 12)}, (f"현재 Python {sys.version.split()[0]}. 런타임 버전을 2026.07(Python 3.12)로 바꾸고 처음부터 실행하세요.")
# Text-only model: remove preinstalled optional packages incompatible with pinned torch.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.5.1", "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(EXPERIMENT / "requirements-colab.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(REPO / "ml")], check=True)

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU(T4) 런타임을 선택하세요"
print(torch.cuda.get_device_name(0), torch.__version__, torch.version.cuda)

In [ ]:
ARM = "f3_balanced_easy"
SEEDS = [42, 43, 44]
assert (RUN / "plan.json").is_file(), "00_prepare_f3를 먼저 완료하세요"
for seed in SEEDS:
    run("train", "--run-dir", RUN, "--arm", ARM, "--seed", seed)
    run("score-windows", "--run-dir", RUN, "--arm", ARM, "--seed", seed)
print("완료:", ARM)

## 다음 단계
끝나면 `02_compare_f3`를 실행합니다.